<a href="https://colab.research.google.com/github/KavallaNikhitha311/Priniples-Of-AI-lab/blob/main/EXP_8%269.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import pandas as pd

class GridWorldMDP:
    def __init__(self, size, goal, trap):
        self.size, self.goal, self.trap = size, goal, trap
        self.state_space = [(i, j) for i in range(size) for j in range(size)]
        self.action_space = ['UP', 'DOWN', 'LEFT', 'RIGHT']
        self.transitions = self.build_transitions()
        self.rewards = self.build_rewards()

    def build_transitions(self):
        return {state: {action: self.calculate_transitions(state, action) for action in self.action_space}
                for state in self.state_space}

    def calculate_transitions(self, state, action):
        i, j = state
        # Handle boundaries and transitions
        if action == 'UP':
            return self.validate_state(i-1, j)
        elif action == 'DOWN':
            return self.validate_state(i+1, j)
        elif action == 'LEFT':
            return self.validate_state(i, j-1)
        elif action == 'RIGHT':
            return self.validate_state(i, j+1)

    def validate_state(self, i, j):
        # Ensure state is within bounds and return the next state
        i, j = max(0, min(i, self.size-1)), max(0, min(j, self.size-1))
        if (i, j) == self.trap:
            return [(1.0, (i, j))]
        return [(1.0, (i, j))]

    def build_rewards(self):
        rewards = {state: -1.0 for state in self.state_space}
        rewards[self.goal] = 0.0
        rewards[self.trap] = -10.0  # Trap has a negative reward
        return rewards

# Value Iteration
def value_iteration(mdp, gamma=0.9, epsilon=0.01):
    state_values = {state: 0.0 for state in mdp.state_space}
    iteration = 0
    print("\nRunning Value Iteration:")

    # Create a DataFrame for better formatting (one row per iteration)
    iteration_results = pd.DataFrame(columns=[str(state) for state in mdp.state_space])

    while True:
        iteration += 1
        delta = 0
        iteration_values = []

        for state in mdp.state_space:
            if state == mdp.goal or state == mdp.trap:
                iteration_values.append(f"{state_values[state]:.2f}")
                continue  # Skip goal and trap states

            v = state_values[state]
            # Calculate the new value using Bellman's equation
            state_values[state] = max(
                sum(p * (mdp.rewards[next_state] + gamma * state_values[next_state])
                    for p, next_state in mdp.transitions[state][action])
                for action in mdp.action_space
            )
            delta = max(delta, abs(v - state_values[state]))  # Track maximum change in values
            iteration_values.append(f"{state_values[state]:.2f}")

        # Add the iteration results to the DataFrame
        iteration_results.loc[iteration] = iteration_values

        # Print iteration results after each round
        print(f"\nIteration {iteration}:")
        print(iteration_results.iloc[iteration-1])

        if delta < epsilon:
            break

    return state_values

# Policy Iteration
def policy_iteration(mdp, gamma=0.9):
    policy = {state: np.random.choice(mdp.action_space) for state in mdp.state_space if state not in [mdp.goal, mdp.trap]}
    state_values = {state: 0.0 for state in mdp.state_space}
    iteration = 0

    while True:
        iteration += 1
        print(f"Policy Iteration - Iteration {iteration}:")

        # Policy Evaluation: Evaluate the current policy
        while True:
            delta = 0
            for state in mdp.state_space:
                if state == mdp.goal or state == mdp.trap:
                    continue
                v = state_values[state]
                # Evaluate the value of each state based on current policy
                state_values[state] = sum(p * (mdp.rewards[next_state] + gamma * state_values[next_state])
                                           for p, next_state in mdp.transitions[state][policy[state]])
                delta = max(delta, abs(v - state_values[state]))
            if delta < 0.01:
                break

        # Print value after each policy evaluation
        print(f"State Values after Policy Evaluation:")
        print(pd.DataFrame.from_dict(state_values, orient='index', columns=["Value"]).T)

        # Policy Improvement: Update the policy
        policy_stable = True
        for state in mdp.state_space:
            if state == mdp.goal or state == mdp.trap:
                continue
            old_action = policy[state]
            # Update policy based on current values
            policy[state] = max(mdp.action_space, key=lambda a: sum(p * (mdp.rewards[next_state] + gamma * state_values[next_state])
                                                                     for p, next_state in mdp.transitions[state][a]))
            if old_action != policy[state]:
                policy_stable = False
        if policy_stable:
            break

        # Print policy after improvement
        print(f"Policy after Iteration {iteration}:")
        print(pd.DataFrame.from_dict(policy, orient='index', columns=["Action"]).T)

    return policy, state_values

# Example Usage (3x3 grid with different goal and trap positions):
size, goal, trap = 3, (2, 2), (1, 1)  # Set a 3x3 grid with a different goal and trap
mdp = GridWorldMDP(size, goal, trap)

# Run Value Iteration
print("\nRunning Value Iteration:")
value_result = value_iteration(mdp)
print("\nFinal Value Iteration Results:")
for state, value in value_result.items():
    print(f"State: {state}, Value: {value:.2f}")

# Run Policy Iteration
print("\nRunning Policy Iteration:")
policy_result, policy_values = policy_iteration(mdp)
print("\nFinal Policy Iteration Results:")
for state, action in policy_result.items():
    print(f"State: {state}, Action: {action}, Value: {policy_values[state]:.2f}")


Running Value Iteration:

Running Value Iteration:

Iteration 1:
(0, 0)    -1.00
(0, 1)    -1.00
(0, 2)    -1.00
(1, 0)    -1.00
(1, 1)     0.00
(1, 2)     0.00
(2, 0)    -1.00
(2, 1)     0.00
(2, 2)     0.00
Name: 1, dtype: object

Iteration 2:
(0, 0)    -1.90
(0, 1)    -1.90
(0, 2)    -1.00
(1, 0)    -1.90
(1, 1)     0.00
(1, 2)     0.00
(2, 0)    -1.00
(2, 1)     0.00
(2, 2)     0.00
Name: 2, dtype: object

Iteration 3:
(0, 0)    -2.71
(0, 1)    -1.90
(0, 2)    -1.00
(1, 0)    -1.90
(1, 1)     0.00
(1, 2)     0.00
(2, 0)    -1.00
(2, 1)     0.00
(2, 2)     0.00
Name: 3, dtype: object

Iteration 4:
(0, 0)    -2.71
(0, 1)    -1.90
(0, 2)    -1.00
(1, 0)    -1.90
(1, 1)     0.00
(1, 2)     0.00
(2, 0)    -1.00
(2, 1)     0.00
(2, 2)     0.00
Name: 4, dtype: object

Final Value Iteration Results:
State: (0, 0), Value: -2.71
State: (0, 1), Value: -1.90
State: (0, 2), Value: -1.00
State: (1, 0), Value: -1.90
State: (1, 1), Value: 0.00
State: (1, 2), Value: 0.00
State: (2, 0), Value: -1.0